# Open-weights rung — reliability probe on Colab (vLLM)

Adds an **open-weights, white-box** model to the panel. It downloads a dense open model, serves it through vLLM's **OpenAI-compatible** endpoint (so the existing OpenAI adapter works with only an env-var swap), and runs a **`near`-only smoke test** as the go/no-go before any full sweep.

**Footguns this notebook handles for you:** YaRN long-context scaling, the tool-call parser (the probe uses native function-calling), and vLLM prefix-caching (the local equivalent of the Anthropic cache trick).

Use a GPU runtime (A100 recommended). On a 40 GB GPU prefer **Qwen3-14B at full precision** over 32B-at-4-bit — the study needs a confound-free measurement, and `near`≈1.0 also guards quantization (if quant broke the model, `near` would drop too).

In [ ]:
# --- config: confirm/swap the model here ---
MODEL = 'Qwen/Qwen3-14B'      # alt: 'Qwen/Qwen3-32B' (needs 4-bit on 40GB), or a newer dense ~27B if available
MODEL_DIR = '/content/model'
SERVED = 'open-model'        # the id the probe passes as --model; must match the served name
MAX_LEN = 131072             # 128k context via YaRN (native window is 32768)
REPO_URL = 'https://github.com/R1ch1k/agent-reliability.git'

In [ ]:
# --- 0. confirm the hardware (decides precision) ---
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
# --- 1. install (vLLM bundles its CUDA/torch deps; this cell is slow on first run) ---
!pip install -q vllm huggingface_hub requests

In [ ]:
# --- 2. download weights explicitly (point local_dir at a mounted Drive path to persist across sessions) ---
from huggingface_hub import snapshot_download
snapshot_download(MODEL, local_dir=MODEL_DIR)   # Qwen3 is Apache-2.0, ungated (no token)
print('downloaded to', MODEL_DIR)

In [ ]:
# --- 3. serve the OpenAI-compatible endpoint in the BACKGROUND, then poll /health ---
import json, subprocess, time, requests
rope = json.dumps({'rope_type': 'yarn', 'factor': 4.0, 'original_max_position_embeddings': 32768})
args = [
    'python', '-m', 'vllm.entrypoints.openai.api_server',
    '--model', MODEL_DIR,
    '--served-model-name', SERVED,
    '--max-model-len', str(MAX_LEN),
    '--rope-scaling', rope,                          # YaRN: without this, >32k is silently broken
    '--enable-auto-tool-choice', '--tool-call-parser', 'hermes',  # the probe uses native tool-calling
    '--enable-prefix-caching',                       # cache the shared manual prefix (fast + cheap)
    '--gpu-memory-utilization', '0.92',
    '--port', '8000',
]
logf = open('/content/vllm.log', 'w')
srv = subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT)
ok = False
for _ in range(240):
    try:
        if requests.get('http://localhost:8000/health').ok:
            ok = True; print('server UP'); break
    except Exception:
        pass
    time.sleep(5)
if not ok:
    print('server did NOT come up — last log lines:')
    print(''.join(open('/content/vllm.log').readlines()[-40:]))

In [ ]:
# --- 4. point the OpenAI SDK at the local server (zero code change) + sanity ping ---
import os
os.environ['OPENAI_BASE_URL'] = 'http://localhost:8000/v1'
os.environ['OPENAI_API_KEY'] = 'EMPTY'   # vLLM needs no auth
from openai import OpenAI
# non-thinking mode for comparability with the API panel (none were run as reasoning models)
r = OpenAI().chat.completions.create(
    model=SERVED,
    messages=[{'role': 'user', 'content': 'reply with the single word OK'}],
    max_tokens=8,
    extra_body={'chat_template_kwargs': {'enable_thinking': False}},
)
print(r.choices[0].message.content)

In [ ]:
# --- 5. clone the harness + run the NEAR-ONLY smoke test (the go/no-go) ---
!git clone -q $REPO_URL /content/probe
%cd /content/probe
# near must hold ~1.0 across the fill range; OPENAI_BASE_URL/KEY from cell 4 propagate to this subprocess
!python reliability_probe_distance.py --provider openai --model $SERVED \
    --conditions near --fills 8000 32000 64000 120000 --runs 5 --needles 3 --depth 0.5

## Go / no-go

- **`near` holds ~1.0 out to 120k** → the capability gate passes; green-light the full sweep.
- **`near` craters at 64–120k** → the usable range is short (or YaRN/quant is degrading it). Cap the sweep there, or raise precision, before spending GPU hours.

Then run the full curve (adds the `distance` condition):

```bash
python reliability_probe_distance.py --provider openai --model open-model \
    --conditions near distance --fills 8000 16000 32000 64000 96000 120000 \
    --runs 15 --needles 3 --depth 0.5
python analyze_curves.py   # places the open model on the R90 ladder
```

**Record for the write-up:** exact model tag, quantization, and KV-cache dtype (the confound). Expect a **mid-ladder** result (Haiku / gpt-4o-mini band), not near Sonnet — open-model effective context sits well below advertised. That's fine: this is the reproducible, white-box rung.